# LeetCode 96: Unique Binary Search Trees

**Difficulty**: Medium  
**Topics**: Dynamic Programming, Tree, Binary Search Tree, Math, Recursion  
**Link**: [LeetCode Problem](https://leetcode.com/problems/unique-binary-search-trees/)

---

## Problem Statement

Given an integer `n`, return the number of structurally unique **BST's** (binary search trees) which has exactly `n` nodes of unique values from `1` to `n`.

### Examples

**Example 1:**
```
Input: n = 3
Output: 5
Explanation: There are 5 unique BSTs with 3 nodes:

   1         3     3      2      1
    \       /     /      / \      \
     3     2     1      1   3      2
    /     /       \                 \
   2     1         2                 3
```

**Example 2:**
```
Input: n = 1
Output: 1
```

### Constraints

- `1 <= n <= 19`

---

## Understanding the Problem

### What Makes BSTs Unique?

Two BSTs are **structurally unique** if they have different shapes, even if they contain the same values.

### Key Insight: Root Selection

For `n` nodes with values `1, 2, 3, ..., n`:
- We can choose **any value** as the root
- If we choose `i` as root:
  - Values `1, 2, ..., i-1` must go in **left subtree** (smaller than root)
  - Values `i+1, i+2, ..., n` must go in **right subtree** (larger than root)

### Visual Example: n = 3

```
Choose root = 1:
    1              Left subtree: 0 nodes (empty)
     \             Right subtree: 2 nodes (2, 3)
    (2,3)          Combinations: 1 × numTrees(2) = 1 × 2 = 2

Choose root = 2:
      2            Left subtree: 1 node (1)
     / \           Right subtree: 1 node (3)
   (1) (3)         Combinations: numTrees(1) × numTrees(1) = 1 × 1 = 1

Choose root = 3:
      3            Left subtree: 2 nodes (1, 2)
     /             Right subtree: 0 nodes (empty)
   (1,2)           Combinations: numTrees(2) × 1 = 2 × 1 = 2

Total: 2 + 1 + 2 = 5
```

### The Recursive Formula

```
numTrees(n) = Σ [numTrees(i-1) × numTrees(n-i)] for i = 1 to n
```

Where:
- `i` is the root value
- `i-1` nodes in left subtree
- `n-i` nodes in right subtree
- Multiply because each left structure can pair with each right structure

### Base Cases

```
numTrees(0) = 1  (empty tree is one valid BST)
numTrees(1) = 1  (single node is one valid BST)
```

---

## Approach 1: Pure Recursion (Naive)

### Algorithm

Directly implement the recursive formula:
1. Base case: if `n = 0` or `n = 1`, return 1
2. For each possible root `i` from 1 to n:
   - Calculate left subtree combinations: `numTrees(i-1)`
   - Calculate right subtree combinations: `numTrees(n-i)`
   - Multiply them and add to total
3. Return total

### Complexity

- **Time**: O(C_n) where C_n is the nth Catalan number ≈ O(4^n / n^(3/2))
  - Exponential! Very slow for large n
- **Space**: O(n) for recursion stack

### Why So Slow?

**Massive overlapping subproblems!**

```
Example: Computing numTrees(4)

                    numTrees(4)
                   /    |    |    \
         T(0)×T(3) T(1)×T(2) T(2)×T(1) T(3)×T(0)
              /         |         |         \
           T(3)       T(2)      T(2)       T(3)
          /  |  \      / \       / \       /  |  \
        ...  ... ...  ... ...  ... ...   ...  ... ...

T(3) computed 2 times
T(2) computed 4 times
T(1) computed 8 times
T(0) computed 8 times

Redundant calculations everywhere!
```

In [ ]:
def numTrees_recursive_naive(n):
    """
    Pure recursion without memoization.
    Time: O(4^n / n^(3/2)) - Exponential!
    Space: O(n) - recursion stack
    """
    # Base cases
    if n <= 1:
        return 1
    
    total = 0
    # Try each value as root
    for i in range(1, n + 1):
        # i is root
        # Left subtree: nodes 1 to i-1 (i-1 nodes)
        # Right subtree: nodes i+1 to n (n-i nodes)
        left = numTrees_recursive_naive(i - 1)
        right = numTrees_recursive_naive(n - i)
        total += left * right
    
    return total

# Test
print("Pure Recursion (Naive):")
for n in range(1, 8):
    result = numTrees_recursive_naive(n)
    print(f"n = {n}: {result} unique BSTs")

print("\nWarning: This is very slow for n > 10!")

## Detailed Recursive Trace

Let's trace the recursion for `n = 3` step by step.

In [ ]:
def numTrees_verbose(n, indent=0, label=""):
    """
    Verbose version showing recursive calls.
    """
    prefix = "  " * indent
    print(f"{prefix}→ numTrees({n}) {label}")
    
    # Base cases
    if n <= 1:
        print(f"{prefix}  Base case: return 1")
        return 1
    
    total = 0
    print(f"{prefix}  Try each root from 1 to {n}:")
    
    for i in range(1, n + 1):
        print(f"{prefix}  Root = {i}:")
        print(f"{prefix}    Left subtree: {i-1} nodes")
        left = numTrees_verbose(i - 1, indent + 2, f"(left of {i})")
        
        print(f"{prefix}    Right subtree: {n-i} nodes")
        right = numTrees_verbose(n - i, indent + 2, f"(right of {i})")
        
        combinations = left * right
        print(f"{prefix}    Combinations: {left} × {right} = {combinations}")
        total += combinations
    
    print(f"{prefix}  Total for numTrees({n}): {total}")
    return total

# Trace for n = 3
print("Detailed Recursive Trace for n = 3:")
print("="*70)
result = numTrees_verbose(3)
print("="*70)
print(f"\nFinal Result: {result}")

## Visualizing the Recursion Tree

In [ ]:
def count_recursive_calls(n, memo=None):
    """
    Count how many times each subproblem is computed.
    """
    if memo is None:
        memo = {}
    
    if n not in memo:
        memo[n] = 0
    memo[n] += 1
    
    if n <= 1:
        return 1, memo
    
    total = 0
    for i in range(1, n + 1):
        left, memo = count_recursive_calls(i - 1, memo)
        right, memo = count_recursive_calls(n - i, memo)
        total += left * right
    
    return total, memo

# Count calls for different n values
print("Recursive Call Count Analysis:\n")
print(f"{'n':<5} {'Result':<10} {'Total Calls':<15} Calls per subproblem")
print("="*70)

for n in range(1, 8):
    result, call_counts = count_recursive_calls(n)
    total_calls = sum(call_counts.values())
    print(f"{n:<5} {result:<10} {total_calls:<15}", end=" ")
    
    # Show distribution
    counts_str = ", ".join([f"T({k}):{v}" for k, v in sorted(call_counts.items())])
    print(counts_str)

print("\nObservation: Subproblems are computed multiple times!")
print("Solution: Use memoization to cache results.")

---

## Approach 2: Recursion with Memoization (Top-Down DP)

### Key Idea

**Cache the results** of subproblems to avoid recomputation.

### Algorithm

1. Create a memo dictionary to store computed results
2. Before computing `numTrees(n)`, check if it's in memo
3. If yes, return cached result
4. If no, compute recursively and store in memo

### Complexity

- **Time**: O(n²)
  - We compute each subproblem (0 to n) exactly once: O(n)
  - Each computation loops through n values: O(n)
  - Total: O(n) × O(n) = O(n²)
- **Space**: O(n) for memo + O(n) for recursion stack = O(n)

### Why Much Faster?

```
Without memoization (n=4):
  T(3) computed 2 times
  T(2) computed 4 times
  T(1) computed 8 times
  T(0) computed 8 times
  Total: 22 calls

With memoization (n=4):
  T(3) computed 1 time
  T(2) computed 1 time
  T(1) computed 1 time
  T(0) computed 1 time
  Total: 5 calls (one per unique subproblem)
```

In [ ]:
def numTrees_memoization(n, memo=None):
    """
    Recursion with memoization (Top-Down DP).
    Time: O(n²)
    Space: O(n)
    """
    # Initialize memo on first call
    if memo is None:
        memo = {}
    
    # Check if already computed
    if n in memo:
        return memo[n]
    
    # Base cases
    if n <= 1:
        return 1
    
    total = 0
    for i in range(1, n + 1):
        left = numTrees_memoization(i - 1, memo)
        right = numTrees_memoization(n - i, memo)
        total += left * right
    
    # Cache result
    memo[n] = total
    return total

# Test
print("Recursion with Memoization:")
for n in range(1, 20):
    result = numTrees_memoization(n)
    print(f"n = {n:2}: {result:6} unique BSTs")

## Memoization Trace with Cache Hits

In [ ]:
def numTrees_memo_verbose(n, memo=None, indent=0, label=""):
    """
    Verbose memoization showing cache hits.
    """
    if memo is None:
        memo = {}
    
    prefix = "  " * indent
    print(f"{prefix}→ numTrees({n}) {label}")
    
    # Check cache
    if n in memo:
        print(f"{prefix}  ✓ CACHE HIT! Return {memo[n]}")
        return memo[n]
    
    # Base cases
    if n <= 1:
        print(f"{prefix}  Base case: return 1")
        return 1
    
    print(f"{prefix}  Computing (not in cache)...")
    total = 0
    
    for i in range(1, n + 1):
        print(f"{prefix}  Root = {i}:")
        left = numTrees_memo_verbose(i - 1, memo, indent + 2, f"(left)")
        right = numTrees_memo_verbose(n - i, memo, indent + 2, f"(right)")
        
        combinations = left * right
        print(f"{prefix}    {left} × {right} = {combinations}")
        total += combinations
    
    print(f"{prefix}  Cache numTrees({n}) = {total}")
    memo[n] = total
    return total

# Trace for n = 4
print("Memoization Trace for n = 4:")
print("="*70)
result = numTrees_memo_verbose(4)
print("="*70)
print(f"\nFinal Result: {result}")
print("\nNotice: Each subproblem computed only once!")

---

## Approach 3: Dynamic Programming (Bottom-Up)

### Key Idea

Build solutions from **smallest to largest** subproblems iteratively.

### Algorithm

1. Create DP array: `dp[i]` = number of unique BSTs with i nodes
2. Initialize base cases: `dp[0] = 1`, `dp[1] = 1`
3. For each `n` from 2 to target:
   - For each possible root `i` from 1 to n:
     - Add `dp[i-1] × dp[n-i]` to `dp[n]`
4. Return `dp[n]`

### Complexity

- **Time**: O(n²)
  - Outer loop: n iterations
  - Inner loop: up to n iterations
  - Total: O(n²)
- **Space**: O(n) for DP array

### Advantages over Memoization

- No recursion overhead
- More space-efficient (no call stack)
- Easier to optimize further
- Iterative = easier to understand execution order

In [ ]:
def numTrees_dp(n):
    """
    Dynamic Programming (Bottom-Up).
    Time: O(n²)
    Space: O(n)
    """
    # dp[i] = number of unique BSTs with i nodes
    dp = [0] * (n + 1)
    
    # Base cases
    dp[0] = 1  # Empty tree
    dp[1] = 1  # Single node
    
    # Build up from 2 to n
    for nodes in range(2, n + 1):
        # Try each value as root
        for root in range(1, nodes + 1):
            left_nodes = root - 1
            right_nodes = nodes - root
            dp[nodes] += dp[left_nodes] * dp[right_nodes]
    
    return dp[n]

# Test
print("Dynamic Programming (Bottom-Up):")
for n in range(1, 20):
    result = numTrees_dp(n)
    print(f"n = {n:2}: {result:6} unique BSTs")

## DP Trace with Table

In [ ]:
def numTrees_dp_verbose(n):
    """
    DP with detailed trace showing table building.
    """
    dp = [0] * (n + 1)
    dp[0] = 1
    dp[1] = 1
    
    print(f"Computing numTrees({n}) using DP:\n")
    print(f"Initial: dp[0] = {dp[0]}, dp[1] = {dp[1]}\n")
    print("="*70)
    
    for nodes in range(2, n + 1):
        print(f"\nComputing dp[{nodes}] (BSTs with {nodes} nodes):")
        print(f"  Try each root from 1 to {nodes}:")
        
        total = 0
        for root in range(1, nodes + 1):
            left_nodes = root - 1
            right_nodes = nodes - root
            combinations = dp[left_nodes] * dp[right_nodes]
            
            print(f"    Root = {root}: left={left_nodes} nodes, right={right_nodes} nodes")
            print(f"             dp[{left_nodes}] × dp[{right_nodes}] = {dp[left_nodes]} × {dp[right_nodes]} = {combinations}")
            
            total += combinations
        
        dp[nodes] = total
        print(f"  dp[{nodes}] = {total}")
        
        # Show current DP table
        print(f"  Current DP table: {dp[:nodes+1]}")
    
    print("\n" + "="*70)
    print(f"\nFinal DP table: {dp}")
    print(f"Result: dp[{n}] = {dp[n]}")
    
    return dp[n]

# Trace for n = 5
result = numTrees_dp_verbose(5)

---

## Approach 4: Catalan Number Formula (Mathematical)

### The Connection to Catalan Numbers

The number of unique BSTs with n nodes is the **nth Catalan number**!

### Catalan Number Definition

The nth Catalan number can be computed using:

```
C(n) = (2n)! / ((n+1)! × n!)
```

Or iteratively:

```
C(0) = 1
C(n) = C(n-1) × 2(2n-1) / (n+1)
```

### Why Catalan Numbers?

Catalan numbers count many combinatorial structures:
- Number of ways to parenthesize expressions
- Number of paths in a grid that don't cross diagonal
- Number of ways to triangulate a polygon
- **Number of structurally unique BSTs** ← Our problem!

### Complexity

- **Time**: O(n)
- **Space**: O(1)

Fastest approach!

In [ ]:
def numTrees_catalan(n):
    """
    Using Catalan number formula.
    Time: O(n)
    Space: O(1)
    """
    if n <= 1:
        return 1
    
    # C(n) = C(n-1) × 2(2n-1) / (n+1)
    catalan = 1
    for i in range(2, n + 1):
        catalan = catalan * 2 * (2 * i - 1) // (i + 1)
    
    return catalan

# Test
print("Catalan Number Formula:")
for n in range(1, 20):
    result = numTrees_catalan(n)
    print(f"n = {n:2}: {result:6} unique BSTs (C_{n-1})")

## Catalan Number Trace

In [ ]:
def numTrees_catalan_verbose(n):
    """
    Catalan formula with step-by-step trace.
    """
    print(f"Computing C({n}) using Catalan formula:\n")
    print("Formula: C(i) = C(i-1) × 2(2i-1) / (i+1)\n")
    
    if n <= 1:
        print(f"Base case: C({n}) = 1")
        return 1
    
    catalan = 1
    print(f"C(1) = {catalan}\n")
    
    for i in range(2, n + 1):
        numerator = 2 * (2 * i - 1)
        denominator = i + 1
        
        print(f"C({i}):")
        print(f"  C({i-1}) × 2(2×{i}-1) / ({i}+1)")
        print(f"  = {catalan} × {numerator} / {denominator}")
        
        catalan = catalan * numerator // denominator
        
        print(f"  = {catalan}\n")
    
    print(f"Result: C({n}) = {catalan}")
    return catalan

# Trace for n = 6
result = numTrees_catalan_verbose(6)

---

## Performance Comparison

In [ ]:
import time

def benchmark_approaches(n):
    """
    Compare performance of different approaches.
    """
    results = {}
    
    # Skip naive recursion for large n (too slow)
    if n <= 10:
        start = time.time()
        result = numTrees_recursive_naive(n)
        elapsed = time.time() - start
        results['Naive Recursion'] = (result, elapsed)
    
    # Memoization
    start = time.time()
    result = numTrees_memoization(n)
    elapsed = time.time() - start
    results['Memoization'] = (result, elapsed)
    
    # DP
    start = time.time()
    result = numTrees_dp(n)
    elapsed = time.time() - start
    results['DP'] = (result, elapsed)
    
    # Catalan
    start = time.time()
    result = numTrees_catalan(n)
    elapsed = time.time() - start
    results['Catalan'] = (result, elapsed)
    
    return results

# Benchmark
print("Performance Comparison:\n")
print(f"{'n':<5} {'Approach':<20} {'Result':<10} {'Time (seconds)'}")
print("="*70)

for n in [5, 10, 15, 19]:
    results = benchmark_approaches(n)
    for approach, (result, elapsed) in results.items():
        print(f"{n:<5} {approach:<20} {result:<10} {elapsed:.6f}")
    print()

print("Observation: Catalan formula is fastest, DP is practical, Memoization is elegant.")

---

## Visualizing All Unique BSTs for Small n

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def generate_trees(n):
    """
    Generate all unique BSTs (LeetCode 95).
    Returns list of root nodes.
    """
    if n == 0:
        return []
    
    def generate(start, end):
        if start > end:
            return [None]
        
        all_trees = []
        for i in range(start, end + 1):
            # Generate all left and right subtrees
            left_trees = generate(start, i - 1)
            right_trees = generate(i + 1, end)
            
            # Combine each left with each right
            for left in left_trees:
                for right in right_trees:
                    root = TreeNode(i)
                    root.left = left
                    root.right = right
                    all_trees.append(root)
        
        return all_trees
    
    return generate(1, n)

def print_tree(root, prefix="", is_tail=True):
    """
    Print tree structure.
    """
    if root is None:
        return
    
    print(prefix + ("└── " if is_tail else "├── ") + str(root.val))
    
    children = []
    if root.left or root.right:
        if root.left:
            children.append((root.left, False))
        if root.right:
            children.append((root.right, True))
    
    for i, (child, is_last) in enumerate(children):
        extension = "    " if is_tail else "│   "
        print_tree(child, prefix + extension, is_last)

# Generate and display all BSTs for n = 3
n = 3
trees = generate_trees(n)
print(f"All {len(trees)} unique BSTs with n = {n}:\n")
print("="*70)

for i, tree in enumerate(trees, 1):
    print(f"\nBST #{i}:")
    print_tree(tree)
    print()

---

## Comparison of All Approaches

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Naive Recursion** | O(4^n) | O(n) | Simple, intuitive | Exponentially slow |
| **Memoization** | O(n²) | O(n) | Elegant, top-down | Recursion overhead |
| **DP** | O(n²) | O(n) | No recursion, clear | More code |
| **Catalan Formula** | O(n) | O(1) | Fastest, minimal space | Requires math knowledge |

### When to Use Each

- **Naive Recursion**: Understanding the problem only
- **Memoization**: Interview (shows DP thinking), elegant code
- **DP**: Production code, clear logic
- **Catalan**: When you recognize the pattern, need optimal performance

---

## Key Takeaways

### Problem Understanding

1. **Structurally unique** means different tree shapes
2. **Root selection** determines left and right subtree sizes
3. **Combinations multiply** because each left pairs with each right

### Recursive Formula

```
numTrees(n) = Σ [numTrees(i-1) × numTrees(n-i)] for i = 1 to n
```

Where:
- `i` = root value
- `i-1` = nodes in left subtree
- `n-i` = nodes in right subtree

### Base Cases

```
numTrees(0) = 1  (empty tree)
numTrees(1) = 1  (single node)
```

### Optimization Journey

1. **Naive Recursion**: O(4^n) - Too slow, massive redundancy
2. **Memoization**: O(n²) - Cache results, eliminate redundancy
3. **DP**: O(n²) - Build bottom-up, no recursion
4. **Catalan**: O(n) - Mathematical formula, optimal

### Why Memoization Works

- **Overlapping subproblems**: Same values computed many times
- **Cache results**: Store in dictionary/array
- **Lookup before compute**: Check cache first
- **Massive speedup**: From exponential to polynomial

### DP Pattern

1. **Define state**: `dp[i]` = answer for i nodes
2. **Base cases**: `dp[0] = 1`, `dp[1] = 1`
3. **Recurrence**: `dp[n] = Σ dp[i-1] × dp[n-i]`
4. **Build order**: Small to large (0 to n)

### Catalan Numbers

- **nth Catalan** = numTrees(n)
- **Formula**: `C(n) = C(n-1) × 2(2n-1) / (n+1)`
- **Applications**: Parenthesization, paths, triangulation, BSTs

### Remember

🎯 **Root selection** creates subproblems  
🎯 **Multiply combinations** (left × right)  
🎯 **Memoization** eliminates redundancy  
🎯 **DP builds bottom-up** from base cases  
🎯 **Catalan numbers** are the pattern  
🎯 **O(n²) is practical**, O(n) is optimal  

Master this problem and you'll understand:
- Recursion with memoization
- Bottom-up DP
- Catalan numbers
- BST combinatorics